In [1]:
# app.py
from flask import Flask, request, render_template, flash, send_file,jsonify, send_from_directory
import numpy as np
from PIL import Image
import torch
from diffusers import StableDiffusionPipeline
import secrets

e:\ProgramsInstallation\anaconda3\envs\tf_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")

True
NVIDIA GeForce RTX 2070 with Max-Q Design


In [3]:
model = "dreamlike-art/dreamlike-diffusion-1.0"

pipe = StableDiffusionPipeline.from_pretrained(model, torch_dtype=torch.float16, use_safetensors=True)
pipe = pipe.to("cuda")

Loading pipeline components...: 100%|██████████| 5/5 [00:02<00:00,  1.78it/s]


In [4]:
try:
    # Load the Keras model
    model = torch.load("E:/CodePractice/Flask/Final API/model.h5")
    print("Model loaded successfully.")
except Exception as e:
    print("Error loading model:", e)


Model loaded successfully.


In [6]:
app = Flask(__name__)
def generate_image(prompt):
    # Assuming `pipe` is defined somewhere in your code
    image = np.array(pipe(prompt=prompt).images[0])
    return image

@app.route('/generate', methods=['POST'])
def generate():
    text = request.json['text']
    image = generate_image(text)
    # Convert the NumPy array to a PIL Image
    pil_image = Image.fromarray(image)
    # Save the PIL Image as a temporary file
    random_string = secrets.token_hex(8)
    filename = f'generated_{random_string}.png'
    pil_image.save(filename)
    # Generate the URL for the image
    url = request.host_url + 'static/' + filename
    # Return the URL as a JSON response
    return jsonify({'image_url': url})

@app.route('/static/<filename>')
def serve_image(filename):
    return send_from_directory('.', filename)

app.run(port=5000)

 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
100%|██████████| 50/50 [00:13<00:00,  3.59it/s]
127.0.0.1 - - [30/May/2024 20:24:23] "POST /generate HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2024 20:24:24] "GET /static/generated_2e74632f4666c9fc.png HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2024 20:24:30] "GET /static/generated_2e74632f4666c9fc.png HTTP/1.1" 200 -
100%|██████████| 50/50 [00:14<00:00,  3.57it/s]
127.0.0.1 - - [30/May/2024 20:25:04] "POST /generate HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2024 20:25:05] "GET /static/generated_780c5f90ef72aebe.png HTTP/1.1" 200 -
127.0.0.1 - - [30/May/2024 20:25:11] "GET /static/generated_780c5f90ef72aebe.png HTTP/1.1" 200 -
